# loading

In [7]:
'''
conda deactivate
conda activate vcc
srun --nodelist=comput76 --pty -c 4 --mem=50G jupyter lab --no-browser --port=8702 --ip=0.0.0.0
ssh 10.168.203.76 -L 8702:127.0.0.1:8702 #-R 37511:localhost:37511
'''

'\nconda deactivate\nconda activate vcc\nsrun --nodelist=comput76 --pty -c 4 --mem=50G jupyter lab --no-browser --port=8702 --ip=0.0.0.0\nssh 10.168.203.76 -L 8702:127.0.0.1:8702 #-R 37511:localhost:37511\n'

In [8]:
import os
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re
%matplotlib inline

In [9]:
from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = 'all'  # 默认为'last'，即输出最后一个结果
InteractiveShell.ast_node_interactivity = 'all'

In [10]:
from openai import OpenAI

In [11]:
import json

In [12]:
import sys
sys.path.append(
    '/public/home/caojun/project/RUSH/2_PxFquery/script')
from PxFquery import pxfquery # NOQA: E402

# 1. data

In [13]:
cp_func_ad = sc.read_h5ad('/public/home/caojun/project/RUSH/3_work/output/store/gsea_anndata/cp_func_ad.h5ad')

In [14]:
sh_func_ad = sc.read_h5ad('/public/home/caojun/project/RUSH/3_work/output/store/gsea_anndata/sh_func_ad.h5ad')

In [15]:
xpr_func_ad = sc.read_h5ad('/public/home/caojun/project/RUSH/3_work/output/store/gsea_anndata/xpr_func_ad.h5ad')

In [19]:
cp_func_ad.obs

,sig_id,project_code,cell_iname,pert_id,cmap_name,pert_dose,pert_time
ABY001_A375_XH:BRD-A90490067:10:24,ABY001_A375_XH:BRD-A90490067:10:24,ABY,A375,BRD-A90490067,fulvestrant,10.0,24.0
ABY001_A375_XH:BRD-K19687926:10:24,ABY001_A375_XH:BRD-K19687926:10:24,ABY,A375,BRD-K19687926,lapatinib,10.0,24.0
ABY001_A375_XH:BRD-K66175015:10:24,ABY001_A375_XH:BRD-K66175015:10:24,ABY,A375,BRD-K66175015,afatinib,10.0,24.0
ABY001_A375_XH:BRD-K70401845:10:24,ABY001_A375_XH:BRD-K70401845:10:24,ABY,A375,BRD-K70401845,erlotinib,10.0,24.0
ABY001_A375_XH:BRD-K70511574:10:24,ABY001_A375_XH:BRD-K70511574:10:24,ABY,A375,BRD-K70511574,HMN-214,10.0,24.0
...,...,...,...,...,...,...,...
TSAI002_MICROGLIA-PSEN1_XH:C646:10,TSAI002_MICROGLIA-PSEN1_XH:C646:10,TSAI,MICROGLIA-PSEN1,C646,C646,10.0,-666.0
TSAI002_MICROGLIA-PSEN1_XH:CI-994:10,TSAI002_MICROGLIA-PSEN1_XH:CI-994:10,TSAI,MICROGLIA-PSEN1,CI-994,CI-994,10.0,-666.0
TSAI002_MICROGLIA-PSEN1_XH:COMPE:2,TSAI002_MICROGLIA-PSEN1_XH:COMPE:2,TSAI,MICROGLIA-PSEN1,COMPE,compe,2.0,-666.0
TSAI002_MICROGLIA-PSEN1_XH:SRT3657:5,TSAI002_MICROGLIA-PSEN1_XH:SRT3657:5,TSAI,MICROGLIA-PSEN1,SRT3657,SRT-3657,5.0,-666.0


In [17]:
sh_func_ad

AnnData object with n_obs × n_vars = 189365 × 91
    obs: 'sig_id', 'project_code', 'cell_iname', 'pert_id', 'cmap_name', 'pert_dose', 'pert_time'
    uns: 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'score', 'score_clip'
    obsp: 'connectivities', 'distances'

In [18]:
xpr_func_ad

AnnData object with n_obs × n_vars = 132464 × 91
    obs: 'sig_id', 'project_code', 'cell_iname', 'pert_id', 'cmap_name', 'pert_dose', 'pert_time'
    uns: 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'score', 'score_clip'
    obsp: 'connectivities', 'distances'

# 2. cell

In [ ]:
#Beam search（束搜索）是一种在“搜索空间很大”时，用有限算力找近似最优解的解码/搜索算法，常见于 机器翻译、语音识别、文本生成（LLM 解码） 等。
#它的核心想法：
#每一步（生成下一个词/动作）不是只保留“当前最好的 1 条路径”（那叫 greedy search），
#也不是把所有可能都展开（那会爆炸），
#而是只保留得分最高的 K 条候选路径继续往下走。这个 K 叫 beam width（束宽）。

In [20]:
cellline_meta_sub = pd.read_csv('/public/home/caojun/project/RUSH/3_work/input/cellline_meta_sub.csv')

In [21]:
cellline_meta_sub

,cell_iname,cell_iname.1,cell_lineage,primary_disease,subtype,cell_alias
0,1HAE,1HAE,unknown,unknown,normal fibroblast sample,NaN
1,AALE,AALE,unknown,unknown,normal epithelium sample,NaN
2,AG06263_2,AG06263_2,unknown,unknown,unknown,NaN
3,AG06840_A,AG06840_A,unknown,unknown,unknown,NaN
4,AG078N1_1,AG078N1_1,unknown,unknown,unknown,NaN
...,...,...,...,...,...,...
235,RCC10RGB,RCC10RGB,kidney,kidney cancer,carcinoma,10RGB
236,OVK18,OVK18,ovary,ovarian cancer,carcinoma,OVK-18
237,JHUEM2,JHUEM2,endometrium,endometrial cancer,carcinoma,JHUEM-2
238,OVCAR8,OVCAR8,ovary,ovarian cancer,carcinoma,OVCAR-8|NIH:OVCAR-8


# 3. drug

# 4. gene

# 5. func

In [ ]:
cp_func_ad.obs['cell_iname'].unique()